In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "your-key-here"
os.environ["LANGSMITH_PROJECT"] = "pr-formal-oleo-17"

In [4]:
!pip install -U ddgs

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.9/862.9 kB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [ddgs]


In [8]:
import os
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain import hub

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    return search.run(query)

llm = ChatAnthropic(model="claude-sonnet-4-6")
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, [web_search], prompt)
executor = AgentExecutor(agent=agent, tools=[web_search], verbose=True)

result = executor.invoke({"input": "What are the latest features announced for LangSmith in 2026?"})
print(result["output"])



> Entering new AgentExecutor chain...
Thought: The user is asking about the latest features announced for LangSmith in 2026. Let me search the web for current information about this.

Action: web_search
Action Input: LangSmith latest features announced 2026
Jan 12, 2026 · To log directly into Gmail, go to the Gmail website on your computer. Enter your Google email and click Next. Then enter your password and click Next. Alternatively, sign into the Google … Email Password Reset Password Log in Don't have an account? Explore Home Features Jan 21, 2026 · To start the sign-in process, open your browser and go to Gmail.com. When the sign-in page appears, enter the email or phone number associated with your Google account. Mar 4, 2026 · Gmail, Google’s email platform, is available on desktop, iPhone, and Android. Signing in is an easy process! Have your Gmail email address and password on hand. This wikiHow will show … Use a private browsing window to sign in. Learn more about using Guest

## Agent 2: Multi-tool router (search + calculator + word count)
Demonstrates tool selection — agent picks the right tool for each sub-task.

In [10]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain import hub
import math

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    return search.run(query)

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression. Example: '2 * (3 + 4)'"""
    try:
        return str(eval(expression, {"__builtins__": {}}, {"math": math}))
    except Exception as e:
        return f"Error: {e}"

@tool
def word_count(text: str) -> str:
    """Count words in a piece of text."""
    return f"{len(text.split())} words"

llm = ChatAnthropic(model="claude-sonnet-4-6")
prompt = hub.pull("hwchase17/react")
tools = [web_search, calculator, word_count]
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

result = executor.invoke({
    "input": "Find Anthropic's latest funding round amount, then calculate what 15% of it would be."
})
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to find information about Anthropic's latest funding round. Let me search for this.

Action: web_search
Action Input: Anthropic latest funding round amount 2024 2025
In November, Nvidia and Microsoft were expected to invest up to $15 billion in Anthropic, and Anthropic said it would buy $30 billion of computing capacity from Microsoft Azure running on Nvidia AI systems. Anthropic is an AI safety and research company that's working to build reliable, interpretable, and steerable AI systems. Claude is Anthropic's AI, built for problem solvers. Tackle complex challenges, analyze data, write code, and think through your hardest work. Apr 7, 2026 · Anthropic, the artificial intelligence company that recently fought the Pentagon over the use of its technology, has built a new A.I. model that it claims is too powerful to be released to the... As part of an accreditation program created for AWS, Anthropic launched a first-of-its-kind trai

## Agent 3: Intentional failures for debugging
Three failure modes a PM needs to handle: bad tool output, infinite loop, malformed query. LangSmith traces each distinctly.

In [11]:
@tool
def broken_database(query: str) -> str:
    """Look up info in the company database."""
    raise ConnectionError("Database connection timed out after 30s")

@tool
def slow_api(query: str) -> str:
    """Query a slow API."""
    return "ERROR: rate limit exceeded, retry in 60s"

@tool
def unreliable_search(query: str) -> str:
    """Search that returns junk."""
    return "asdfqwerty random garbage data 42 banana"

llm = ChatAnthropic(model="claude-sonnet-4-6")
prompt = hub.pull("hwchase17/react")
tools = [broken_database, slow_api, unreliable_search]
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    max_iterations=4,
    handle_parsing_errors=True
)

result = executor.invoke({
    "input": "Look up customer ID 12345 in the database and tell me their order history."
})
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to look up customer ID 12345 in the company database to find their order history.
Action: broken_database
Action Input: customer ID 12345 order history


ConnectionError: Database connection timed out after 30s

## Agent 3b: Graceful failure handling
Same scenario, but tools return error strings instead of raising. Agent can now reason about failures and attempt recovery.

In [12]:
@tool
def broken_database(query: str) -> str:
    """Look up info in the company database."""
    return "ERROR: Database connection timed out after 30s"

@tool
def slow_api(query: str) -> str:
    """Query a slow API."""
    return "ERROR: rate limit exceeded, retry in 60s"

@tool
def unreliable_search(query: str) -> str:
    """Search that returns junk."""
    return "asdfqwerty random garbage data 42 banana"

llm = ChatAnthropic(model="claude-sonnet-4-6")
prompt = hub.pull("hwchase17/react")
tools = [broken_database, slow_api, unreliable_search]
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    max_iterations=4,
    handle_parsing_errors=True
)

result = executor.invoke({
    "input": "Look up customer ID 12345 in the database and tell me their order history."
})
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to look up customer ID 12345 in the company database to find their order history.
Action: broken_database
Action Input: customer ID 12345 order history
ERROR: Database connection timed out after 30sThought: The database connection timed out. Let me try again to see if it was a temporary issue.
Action: broken_database
Action Input: customer ID 12345 order history
ERROR: Database connection timed out after 30sThought: The database is consistently timing out. Let me try the slow API as an alternative to retrieve customer information for ID 12345.
Action: slow_api
Action Input: customer ID 12345 order history
ERROR: rate limit exceeded, retry in 60sThought: The slow API is also unavailable due to rate limiting. Let me try the unreliable search as a last resort to find information about customer ID 12345.
Action: unreliable_search
Action Input: customer ID 12345 order history
asdfqwerty random garbage data 42 banana

> Finished chain.


In [ ]:
os.environ["LANGSMITH_API_KEY"] = "your-key-here"

## Agent 4: Evaluation on a dataset
Run the weather agent against 5 test cases, score each output programmatically. Foundation for scaling to 10k tests.

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client(api_key="your-key-here")

dataset_name = "weather-agent-eval"
try:
    dataset = client.create_dataset(dataset_name)
    examples = [
        {"input": "What's the weather in Tokyo?", "expected_city": "Tokyo"},
        {"input": "How's San Francisco looking today?", "expected_city": "San Francisco"},
        {"input": "Tell me about weather in Paris", "expected_city": "Paris"},
        {"input": "Mumbai weather please", "expected_city": "Mumbai"},
        {"input": "What about London?", "expected_city": "London"},
    ]
    for ex in examples:
        client.create_example(
            inputs={"input": ex["input"]},
            outputs={"expected_city": ex["expected_city"]},
            dataset_id=dataset.id,
        )
except Exception as e:
    print(f"Dataset may already exist: {e}")

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

llm = ChatAnthropic(model="claude-sonnet-4-6")
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, [get_weather], prompt)
executor = AgentExecutor(agent=agent, tools=[get_weather], verbose=False, handle_parsing_errors=True)

def predict(inputs: dict) -> dict:
    result = executor.invoke({"input": inputs["input"]})
    return {"output": result["output"]}

def city_mentioned(outputs: dict, reference_outputs: dict) -> bool:
    return reference_outputs["expected_city"].lower() in outputs["output"].lower()

results = evaluate(
    predict,
    data=dataset_name,
    evaluators=[city_mentioned],
    experiment_prefix="weather-agent-v1",
    client=client,
)
print("Done — check LangSmith Datasets & Experiments tab")

/Users/siddharthrallabandi/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'weather-agent-v1-e1882a68' at:
https://smith.langchain.com/o/21c19e8b-548d-420b-9300-7cfda405ae98/datasets/db338c82-65c4-4440-9062-6609b137088b/compare?selectedSessions=2032e8f7-bab6-460c-bf3e-612c49963f1e




5it [00:21,  4.24s/it]

Done — check LangSmith Datasets & Experiments tab


In [20]:
import os
print("Key:", os.environ.get("LANGSMITH_API_KEY", "NOT SET")[:20] + "...")
print("Endpoint:", os.environ.get("LANGSMITH_ENDPOINT", "NOT SET"))

Key: lsv2_pt_74d53213e6f1...
Endpoint: NOT SET


In [27]:
from langsmith import Client
c = Client(api_key="lsv2_pt_PASTE_FULL_KEY_HERE")
print(c.list_datasets(limit=1))

<generator object Client.list_datasets at 0x1170d6ac0>


## Agent 5: Prompt A/B testing
Run the same dataset against two different prompt variants. LangSmith compares them side-by-side.

In [31]:
from langchain.prompts import PromptTemplate

# Variant A: terse
prompt_a_text = """Answer briefly. Use tools when needed.

Tools: {tools}
Tool names: {tool_names}

Question: {input}
Thought:{agent_scratchpad}"""

# Variant B: verbose/friendly
prompt_b_text = """You are a friendly, enthusiastic weather assistant. Always greet the user warmly and add weather-related emojis.

You have access to these tools:
{tools}

Use this format:
Question: the input question
Thought: think about what to do
Action: the action to take, one of [{tool_names}]
Action Input: input for the action
Observation: action result
... (repeat as needed)
Thought: I now know the answer
Final Answer: the final answer

Question: {input}
Thought:{agent_scratchpad}"""

def run_variant(prompt_text, variant_name):
    p = PromptTemplate.from_template(prompt_text)
    agent = create_react_agent(llm, [get_weather], p)
    exec_ = AgentExecutor(agent=agent, tools=[get_weather], handle_parsing_errors=True, verbose=False)
    
    def predict(inputs):
        return {"output": exec_.invoke({"input": inputs["input"]})["output"]}
    
    return evaluate(
        predict,
        data=dataset_name,
        evaluators=[city_mentioned],
        experiment_prefix=f"weather-{variant_name}",
        client=client,
    )

print("Running Variant A (terse)...")
run_variant(prompt_a_text, "variant-a-terse")
print("Running Variant B (friendly)...")
run_variant(prompt_b_text, "variant-b-friendly")
print("Done — compare both in LangSmith Experiments tab")

Running Variant A (terse)...
View the evaluation results for experiment: 'weather-variant-a-terse-88127062' at:
https://smith.langchain.com/o/21c19e8b-548d-420b-9300-7cfda405ae98/datasets/db338c82-65c4-4440-9062-6609b137088b/compare?selectedSessions=5752a36c-aceb-447c-89c9-e228725efde7




5it [00:43,  8.79s/it]


Running Variant B (friendly)...
View the evaluation results for experiment: 'weather-variant-b-friendly-1e5d0a3d' at:
https://smith.langchain.com/o/21c19e8b-548d-420b-9300-7cfda405ae98/datasets/db338c82-65c4-4440-9062-6609b137088b/compare?selectedSessions=163419e4-a639-44e4-a233-bee89298acc5




5it [00:27,  5.41s/it]

Done — compare both in LangSmith Experiments tab


# PM Career Co-Pilot (Self Agent v1)
Reads Gmail, Notion, and Google Drive. Returns a structured "where am I in my AI PM journey" briefing.

In [32]:
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client notion-client pydantic

Python(6866) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [33]:
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/drive.readonly",
]

def get_google_creds():
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                os.path.expanduser("~/credentials.json"), SCOPES
            )
            creds = flow.run_local_server(port=0)
        with open("token.json", "w") as f:
            f.write(creds.to_json())
    return creds

google_creds = get_google_creds()
gmail = build("gmail", "v1", credentials=google_creds)
drive = build("drive", "v3", credentials=google_creds)
print("✅ Gmail + Drive authenticated")

/Users/siddharthrallabandi/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/siddharthrallabandi/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/siddharthrallabandi/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Pyth

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=953505929222-oo46t0obv4bv25fl19j2s6sce6mv9g42.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A56421%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=4645EWEpC8xobVU6BLJDoRw5HwrSC9&code_challenge=qpmL19KG9F-fGINOrd7gvm_prmh86Y27w2FKzN5E0k0&code_challenge_method=S256&access_type=offline
✅ Gmail + Drive authenticated


In [34]:
profile = gmail.users().getProfile(userId="me").execute()
print(f"✅ Gmail: {profile['emailAddress']}")

files = drive.files().list(pageSize=3, fields="files(name)").execute()
print(f"✅ Drive: {[f['name'] for f in files.get('files', [])]}")


✅ Gmail: rallabandisiddharth@gmail.com
✅ Drive: ['AI Agent Strategist Role Discussion', 'Deep Dive into Pinecone Product Management Role', 'Agentic AI Product Strategy Research']


In [ ]:
from notion_client import Client as NotionClient

NOTION_TOKEN = "your-key-here"  # starts with ntn_ or secret_

notion = NotionClient(auth=NOTION_TOKEN)

# Test: list pages the integration has access to
results = notion.search(filter={"property": "object", "value": "page"}).get("results", [])
print(f"✅ Notion: {len(results)} pages accessible")
for r in results[:5]:
    title = r.get("properties", {}).get("title", {}).get("title", [])
    name = title[0]["plain_text"] if title else r.get("url", "untitled")
    print(f"  - {name}")

✅ Notion: 3 pages accessible
  - Siddharth Rallabandi | Forward-Deployed AI Product Leader
  - AI Prompt Library & Context Engineering Lab - Siddharth Rallabandi
  - Vibe Coding Prompts


In [37]:
from langchain.tools import tool
from email.mime.text import MIMEText
import base64

@tool
def search_gmail(query: str) -> str:
    """Search Gmail for messages matching a query. Use Gmail search syntax (e.g., 'from:recruiter newer_than:7d', 'subject:interview'). Returns up to 10 message subjects + snippets."""
    try:
        results = gmail.users().messages().list(userId="me", q=query, maxResults=10).execute()
        messages = results.get("messages", [])
        if not messages:
            return "No messages found."
        out = []
        for m in messages:
            msg = gmail.users().messages().get(userId="me", id=m["id"], format="metadata", metadataHeaders=["Subject", "From", "Date"]).execute()
            headers = {h["name"]: h["value"] for h in msg["payload"]["headers"]}
            snippet = msg.get("snippet", "")[:200]
            out.append(f"From: {headers.get('From','?')}\nSubject: {headers.get('Subject','?')}\nDate: {headers.get('Date','?')}\nSnippet: {snippet}\n---")
        return "\n".join(out)
    except Exception as e:
        return f"Error: {e}"

@tool
def list_drive_files(query: str) -> str:
    """Search Google Drive for files. Query is a name keyword (e.g., 'resume', 'cover letter'). Returns file names and IDs."""
    try:
        results = drive.files().list(q=f"name contains '{query}' and trashed=false", pageSize=10, fields="files(id, name, mimeType, modifiedTime)").execute()
        files = results.get("files", [])
        if not files:
            return "No files found."
        return "\n".join([f"{f['name']} (id: {f['id']}, modified: {f['modifiedTime']})" for f in files])
    except Exception as e:
        return f"Error: {e}"

@tool
def read_drive_file(file_id: str) -> str:
    """Read the contents of a Google Drive file by its ID. Works for Google Docs and plain text files."""
    try:
        meta = drive.files().get(fileId=file_id, fields="mimeType,name").execute()
        if meta["mimeType"] == "application/vnd.google-apps.document":
            content = drive.files().export(fileId=file_id, mimeType="text/plain").execute()
            return content.decode("utf-8")[:5000]
        else:
            content = drive.files().get_media(fileId=file_id).execute()
            return content.decode("utf-8", errors="ignore")[:5000]
    except Exception as e:
        return f"Error: {e}"

@tool
def search_notion(query: str) -> str:
    """Search Notion pages for content matching a query. Returns page titles and IDs."""
    try:
        results = notion.search(query=query).get("results", [])
        if not results:
            return "No Notion pages found."
        out = []
        for r in results[:10]:
            title_arr = r.get("properties", {}).get("title", {}).get("title", []) or r.get("properties", {}).get("Name", {}).get("title", [])
            name = title_arr[0]["plain_text"] if title_arr else "untitled"
            out.append(f"{name} (id: {r['id']})")
        return "\n".join(out)
    except Exception as e:
        return f"Error: {e}"

@tool
def read_notion_page(page_id: str) -> str:
    """Read the contents of a Notion page by its ID."""
    try:
        blocks = notion.blocks.children.list(block_id=page_id).get("results", [])
        text = []
        for b in blocks:
            t = b.get("type")
            content = b.get(t, {}).get("rich_text", [])
            line = "".join([c.get("plain_text", "") for c in content])
            if line:
                text.append(line)
        return "\n".join(text)[:5000] or "Page is empty."
    except Exception as e:
        return f"Error: {e}"

print("✅ 5 tools defined: search_gmail, list_drive_files, read_drive_file, search_notion, read_notion_page")

✅ 5 tools defined: search_gmail, list_drive_files, read_drive_file, search_notion, read_notion_page


In [38]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

tools = [search_gmail, list_drive_files, read_drive_file, search_notion, read_notion_page]

# Custom prompt tuned for PM career analysis
from langchain.prompts import PromptTemplate

self_agent_prompt = PromptTemplate.from_template("""You are a Career Co-Pilot helping Siddharth transition from AI Platform Engineering to AI Product Management roles at Anthropic, OpenAI, Perplexity, LangChain, and similar companies.

You have access to his Gmail, Google Drive, and Notion workspace. Your job is to gather signal from his data and synthesize it into honest, structured insights — not generic advice.

Principles:
- Be specific. Cite actual emails, file names, or Notion pages.
- Be honest about gaps. If something looks weak, say so.
- Be efficient. Don't read everything — search precisely, then read only what matters.
- Flag uncertainty. If the data is thin, say "limited signal here" rather than making things up.

Available tools:
{tools}

Use this format:
Question: the input question
Thought: think step-by-step about what data you need and which tool to call
Action: one of [{tool_names}]
Action Input: input for the tool
Observation: tool result
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough to answer
Final Answer: structured response with specific findings

Question: {input}
Thought:{agent_scratchpad}""")

self_agent = create_react_agent(llm, tools, self_agent_prompt)
self_executor = AgentExecutor(
    agent=self_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
)

print("✅ Self Agent ready")

✅ Self Agent ready


In [39]:
result = self_executor.invoke({
    "input": "Give me a brief 'state of my AI PM job search' report. Check: (1) recent recruiter emails or interview activity in Gmail (last 14 days), (2) any resume or cover letter files in Drive, (3) my AI Prompt Library in Notion. Then synthesize: where am I strong, where are the gaps, and what are the top 3 actions for this week."
})
print("\n\n=== FINAL REPORT ===\n")
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to gather data from three sources systematically. Let me start with Gmail to check recent recruiter/interview activity, then Drive for resume/cover letter files, then Notion for the AI Prompt Library. I'll search precisely rather than broadly.

Action: search_gmail
Action Input: recruiter OR interview OR "product manager" OR "PM role" OR "job opportunity" newer_than:14d
From: Harvey Hiring Team <no-reply@ashbyhq.com>
Subject: Thank You for Applying to Harvey
Date: Sat, 9 May 2026 22:14:57 +0000
Snippet: Hi Siddharth, Thank you for taking the time to apply to the Staff Product Manager, Agent Platform role here at Harvey. We truly appreciate the time and effort you put into your application. Our hiring
---
From: no-reply@waymo.com
Subject: Update from Waymo
Date: Fri, 08 May 2026 17:42:51 +0000
Snippet: Hi Siddharth, Thank you for your interest in Waymo and giving us the opportunity to review your experience and qualifications. We r

In [40]:
r1 = self_executor.invoke({
    "input": "Check Gmail for the last 14 days of job search activity. Categorize results into: (a) rejections, (b) in-progress/active conversations, (c) new inbound opportunities, (d) my own outbound applications. Be specific — name the company for each. Return a structured bullet list."
})
print("\n=== GMAIL ===\n", r1["output"])



> Entering new AgentExecutor chain...
Thought: I need to search Gmail for job search related emails from the last 14 days. I'll run several targeted searches to capture different categories: rejections, recruiter conversations, inbound opportunities, and outbound applications. Let me start broad and then get specific.

Action: search_gmail
Action Input: subject:(application OR interview OR opportunity OR role OR position) newer_than:14d
From: Pinecone Hiring Team <no-reply@ashbyhq.com>
Subject: Your application to Pinecone
Date: Wed, 6 May 2026 13:04:56 +0000
Snippet: Hi Siddharth, Thank you for taking the time to apply for the Staff/Principal Product Manager, Database position at Pinecone. We&#39;re sorry to inform you that we have decided to move forward with
---
From: "Notion's Recruiting Team" <recruiting-no-reply@makenotion.com>
Subject: Update on your application to Notion, Siddharth
Date: Thu, 30 Apr 2026 13:09:48 +0000
Snippet: Hi Siddharth, Thanks for your thoughtful applica

In [41]:
r2 = self_executor.invoke({
    "input": "Find my resume and cover letter files in Google Drive. Read the most recent resume. Tell me: (1) what version/date it is, (2) what AI PM-relevant strengths it highlights, (3) what's missing or weak for an AI PM role at companies like Anthropic or LangChain."
})
print("\n=== DRIVE ===\n", r2["output"])



> Entering new AgentExecutor chain...
Thought: I need to search Google Drive for resume and cover letter files first, then read the most recent resume to analyze it.

Action: list_drive_files
Action Input: resume
Error: [Errno 54] Connection reset by peerThought: The first attempt failed with a connection error. Let me retry the Drive search for resume files.

Action: list_drive_files
Action Input: resume
AI Product Manager Resume Analysis (id: 1yGTLZsW4Q4hJ59yX2B2uMWdYtei-XmoGwmvoCXXHbgA, modified: 2026-03-30T06:16:05.365Z)
AI Product Manager Resume Analysis - Stripe (id: 1eus_6eoLQei0ggluXeOqg_-9t67awlBOW8UkVJbwHOA, modified: 2026-03-30T06:11:12.447Z)Thought: I found two resume-related files. The most recent one is "AI Product Manager Resume Analysis" (modified 2026-03-30). Let me read that file first, then also check for cover letter files.

Action: read_drive_file
Action Input: 1yGTLZsW4Q4hJ59yX2B2uMWdYtei-XmoGwmvoCXXHbgA
Error: <HttpError 404 when requesting https://www.googleap

In [45]:
@tool
def read_notion_page(page_id: str) -> str:
    """Read the contents of a Notion page by its ID."""
    page_id = page_id.strip()
    try:
        blocks = notion.blocks.children.list(block_id=page_id).get("results", [])
        text = []
        for b in blocks:
            t = b.get("type")
            content = b.get(t, {}).get("rich_text", [])
            line = "".join([c.get("plain_text", "") for c in content])
            if line:
                text.append(line)
        return "\n".join(text)[:5000] or "Page is empty."
    except Exception as e:
        return f"Error: {e}"

In [47]:
@tool
def read_notion_page(page_id: str) -> str:
    """Read the contents of a Notion page by its ID."""
    page_id = page_id.strip().replace("\n", "").replace(" ", "")
    try:
        blocks = notion.blocks.children.list(block_id=page_id).get("results", [])
        text = []
        for b in blocks:
            t = b.get("type")
            content = b.get(t, {}).get("rich_text", [])
            line = "".join([c.get("plain_text", "") for c in content])
            if line:
                text.append(line)
        return "\n".join(text)[:5000] or "Page is empty."
    except Exception as e:
        return f"Error: {e}"

@tool
def read_drive_file(file_id: str) -> str:
    """Read the contents of a Google Drive file by its ID. Works for Google Docs and plain text files."""
    file_id = file_id.strip().replace("\n", "").replace(" ", "")
    try:
        meta = drive.files().get(fileId=file_id, fields="mimeType,name").execute()
        if meta["mimeType"] == "application/vnd.google-apps.document":
            content = drive.files().export(fileId=file_id, mimeType="text/plain").execute()
            return content.decode("utf-8")[:5000]
        else:
            content = drive.files().get_media(fileId=file_id).execute()
            return content.decode("utf-8", errors="ignore")[:5000]
    except Exception as e:
        return f"Error: {e}"

# REBUILD agent with updated tools
tools = [search_gmail, list_drive_files, read_drive_file, search_notion, read_notion_page]
self_agent = create_react_agent(llm, tools, self_agent_prompt)
self_executor = AgentExecutor(
    agent=self_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=15,
)
print("✅ Agent rebuilt with sanitized tools")

✅ Agent rebuilt with sanitized tools


In [48]:
r3 = self_executor.invoke({
    "input": "Look at my 'AI Prompt Library & Context Engineering Lab' Notion page. Summarize: (1) what's actually in there, (2) is it portfolio-ready for an AI PM application, (3) what are the top 3 things missing."
})
print("\n=== NOTION ===\n", r3["output"])



> Entering new AgentExecutor chain...
Thought: I need to find and read the 'AI Prompt Library & Context Engineering Lab' Notion page. Let me search for it first.

Action: search_notion
Action Input: AI Prompt Library Context Engineering Lab
Siddharth Rallabandi | Forward-Deployed AI Product Leader (id: 2cd98324-160d-80ce-bf74-cf0c0a3af0f5)
AI Prompt Library & Context Engineering Lab - Siddharth Rallabandi (id: 31f98324-160d-80de-8b07-dee17cebc9d1)
Vibe Coding Prompts (id: 33998324-160d-802d-b47c-c076777e45f5)Thought: I found the page. Let me read it now.

Action: read_notion_page
Action Input: 31f98324-160d-80de-8b07-dee17cebc9d1
Operating Philosophy: A frontier model is only as effective as the context it is grounded in. This database catalogs my production-tested prompts, context bounding strategies, and adversarial evaluations used to ship enterprise-grade AI features, automate infrastructure, and accelerate product velocity.
Updated total: 73 prompts across 15 categories (includi